**MGMT298D: Science and Strategy of AI**
# Week 5: Convolutional Filters

#### This notebook explores what convolutional layers actually learn by visualizing the internal activations and filter weights of a pretrained VGG16 network at different depths.

# 1 Setup

#### We load a pretrained VGG16 from `keras`, upload an image, resize it to 224×224, and confirm the model classifies it correctly before inspecting its internals.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

In [ ]:
# Upload your own image, or press Cancel to use the default (Kitty.jpeg)
from google.colab import files
try:
    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]
except Exception:
    import urllib.request
    url = 'https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/Kitty.jpeg'
    img_path = 'Kitty.jpeg'
    urllib.request.urlretrieve(url, img_path)
    print(f'Using default image: {img_path}')

# Load and preprocess for VGG16 (224x224, BGR, centered)
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_batch = preprocess_input(np.expand_dims(img_array, axis=0))

# Load pretrained VGG16
model = VGG16(weights='imagenet')

# Show the image
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title('Input Image (224×224)', fontsize=11)
plt.axis('off')
plt.show()

In [ ]:
# Confirm correct classification
preds = model.predict(img_batch, verbose=0)
top3 = decode_predictions(preds, top=3)[0]
for label, name, prob in top3:
    print(f'{name:>25s}: {prob:.1%}')

In [ ]:
# VGG16 architecture: 5 conv blocks, each with 2-3 conv layers + max pooling
for i, layer in enumerate(model.layers):
    print(f'{i:2d}  {layer.name:20s}  output: {layer.output.shape}')

---
# 2 Feature Maps

#### We build a model that outputs activations at every convolutional layer, then visualize what early, middle, and deep layers detect, from simple edges up to complex object parts.

In [ ]:
# Extract all Conv2D layer activations in one forward pass
conv_layers = [l for l in model.layers if 'conv' in l.name]
conv_names = [l.name for l in conv_layers]

activation_model = Model(inputs=model.input, outputs=[l.output for l in conv_layers])
activations = activation_model.predict(img_batch, verbose=0)

print(f'{len(conv_layers)} convolutional layers extracted')
for name, act in zip(conv_names, activations):
    print(f'  {name:16s}  shape: {act.shape[1]}×{act.shape[2]}, {act.shape[3]} filters')

In [ ]:
# Early layer (block1_conv1): 64 filters detecting edges, gradients, color contrasts
act1 = activations[0][0]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('Block 1, Conv 1, Edge Detectors (first 16 of 64 filters)', fontsize=13, y=1.01)
for i, ax in enumerate(axes.flat):
    ax.imshow(act1[:, :, i], cmap='viridis')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Middle layer (block3_conv3): curves, textures, repeating structures
idx_mid = conv_names.index('block3_conv3')
act_mid = activations[idx_mid][0]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle(f'Block 3, Conv 3, Texture/Pattern Detectors ({act_mid.shape[2]} filters, {act_mid.shape[0]}×{act_mid.shape[1]} spatial)', fontsize=13, y=1.01)
for i, ax in enumerate(axes.flat):
    ax.imshow(act_mid[:, :, i], cmap='viridis')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Deep layer (block5_conv3): high-level concepts at 14x14 spatial resolution
idx_deep = conv_names.index('block5_conv3')
act_deep = activations[idx_deep][0]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle(f'Block 5, Conv 3, High-Level Detectors ({act_deep.shape[2]} filters, {act_deep.shape[0]}×{act_deep.shape[1]} spatial)', fontsize=13, y=1.01)
for i, ax in enumerate(axes.flat):
    ax.imshow(act_deep[:, :, i], cmap='viridis')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
# 3 All Layers at a Glance

#### One representative filter from each convolutional layer, arranged left to right. The representation progresses from pixel-level edges to abstract object-level patterns, and the spatial size shrinks at each pooling boundary.

In [ ]:
# Pick the highest-activation filter from each layer
fig, axes = plt.subplots(2, 7, figsize=(18, 5))
fig.suptitle('One Filter Per Layer, From Edges to Objects', fontsize=14, y=1.03)

for i, (name, act) in enumerate(zip(conv_names[:13], activations[:13])):
    row, col = i // 7, i % 7
    fmap = act[0]
    best_filter = fmap.mean(axis=(0, 1)).argmax()
    axes[row, col].imshow(fmap[:, :, best_filter], cmap='inferno')
    axes[row, col].set_title(f'{name}\n{fmap.shape[0]}×{fmap.shape[1]}', fontsize=8)
    axes[row, col].axis('off')

if len(conv_names) < 14:
    axes[1, 6].axis('off')

plt.tight_layout()
plt.show()

---
# 4 Impact of Pooling

#### Max pooling keeps only the strongest activation in each 2×2 window, reducing spatial dimensions. We show the same feature map before and after each pooling step.

In [ ]:
# Activations before and after each max pool
pool_pairs = [
    ('block1_conv2', 'block1_pool'),
    ('block2_conv2', 'block2_pool'),
    ('block3_conv3', 'block3_pool'),
]

pool_model = Model(
    inputs=model.input,
    outputs=[model.get_layer(n).output for pair in pool_pairs for n in pair]
)
pool_acts = pool_model.predict(img_batch, verbose=0)

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle('Before vs After Max Pooling, Same Filter', fontsize=14, y=1.01)

for row, (before_name, after_name) in enumerate(pool_pairs):
    before = pool_acts[row * 2][0]
    after = pool_acts[row * 2 + 1][0]
    filt = before.mean(axis=(0, 1)).argmax()

    axes[row, 0].imshow(before[:, :, filt], cmap='viridis')
    axes[row, 0].set_title(f'{before_name} ({before.shape[0]}×{before.shape[1]})', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(after[:, :, filt], cmap='viridis')
    axes[row, 1].set_title(f'{after_name} ({after.shape[0]}×{after.shape[1]})', fontsize=10)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()